# Exercise 13 — Transactional Pattern Mining of PM2.5 Air Pollution Data


**GitHub Repository (forked PAMI):** [https://github.com/KaswanDevkishan/PAMI](https://github.com/KaswanDevkishan/PAMI)

---

## Overview

In Exercise 10 we built a dense data frame `pm25_sensor_data.csv` in the format:

```
Timestamp, Point(sensorID_1), Point(sensorID_2), ...
```

In this exercise we:

1. Load and clean that data frame.
2. Convert it into a **transactional database** using the PAMI library, where a "transaction" is the set of sensors that recorded a heavily-polluted (PM2.5 ≥ 15) reading at a given timestamp.
3. Mine **frequent patterns** (sets of sensors that are frequently polluted *together*) with the **FP-Growth** algorithm.
4. Identify the **longest frequent pattern** and visualize those sensor locations on an OpenStreetMap using `plotly.express`.


## 0. Setup (Google Colab)

Run this cell first. It installs PAMI and (if you are on Colab) lets you upload the two input files:

- `pm25_sensor_data.csv` — produced in Exercise 10
- `stationInfo.csv` — station ID → latitude/longitude master file (needed later, for the map)


In [ ]:
# Install the PAMI library (pattern mining) and plotly (visualization)
!pip install -U PAMI plotly --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.0 MB/s eta 0:00:00


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Please upload 'pm25_sensor_data.csv' and 'stationInfo.csv'")
    uploaded = files.upload()
else:
    # Running locally / outside Colab: make sure these two files are
    # already present in the working directory.
    print("Not running in Colab — make sure pm25_sensor_data.csv and "
          "stationInfo.csv are already in the working directory.")


Please upload 'pm25_sensor_data.csv' and 'stationInfo.csv'


Saving pm25_sensor_data.csv to pm25_sensor_data.csv
Saving stationInfo.csv to stationInfo.csv


## 1.a) Read the Exercise-10 CSV into a DataFrame

In [ ]:
import pandas as pd

df = pd.read_csv('pm25_sensor_data.csv')

print("Shape (timestamps x (sensors+1)):", df.shape)
df.head()


Shape (timestamps x (sensors+1)): (720, 1084)


,Timestamp,Point(01101520),Point(01101540),Point(01102010),Point(01102020),Point(01102510),Point(01105520),Point(01107020),Point(01107030),Point(01110010),...,Point(46208010),Point(46212010),Point(46220010),Point(46225010),Point(46321010),Point(47201140),Point(47206020),Point(47209080),Point(47211050),Point(47301950)
0,2026/06/01 01:00:00,6.0,7.0,7.0,4.0,7.0,5.0,4.0,6.0,5.0,...,15.0,9.0,9.0,13.0,9.0,NaN,NaN,NaN,NaN,15.0
1,2026/06/01 02:00:00,8.0,5.0,9.0,1.0,3.0,6.0,6.0,5.0,5.0,...,14.0,10.0,9.0,13.0,8.0,NaN,NaN,NaN,NaN,17.0
2,2026/06/01 03:00:00,4.0,5.0,9.0,3.0,6.0,4.0,5.0,6.0,6.0,...,13.0,10.0,9.0,12.0,8.0,NaN,NaN,NaN,NaN,18.0
3,2026/06/01 04:00:00,4.0,3.0,10.0,1.0,6.0,3.0,6.0,7.0,4.0,...,10.0,10.0,9.0,10.0,10.0,NaN,NaN,NaN,NaN,16.0
4,2026/06/01 05:00:00,6.0,5.0,10.0,-2.0,5.0,4.0,6.0,6.0,4.0,...,9.0,8.0,8.0,10.0,9.0,NaN,NaN,NaN,NaN,17.0


## 1.b) Data preprocessing

### i) Replace NaN values with 0
### ii) Replace values ≥ 100 with 0 (sensor/measurement errors)
### iii) Remove the `Timestamp` column


In [ ]:
# i) Replace NaN -> 0
df_clean = df.fillna(0)

# ii) Replace values >= 100 -> 0
sensor_cols = [c for c in df_clean.columns if c != 'Timestamp']
df_clean[sensor_cols] = df_clean[sensor_cols].mask(df_clean[sensor_cols] >= 100, 0)

# iii) Remove the "Timestamp" column
df_clean = df_clean.drop(columns=['Timestamp'])

print("Cleaned shape:", df_clean.shape)
df_clean.head()


Cleaned shape: (720, 1083)


,Point(01101520),Point(01101540),Point(01102010),Point(01102020),Point(01102510),Point(01105520),Point(01107020),Point(01107030),Point(01110010),Point(01202080),...,Point(46208010),Point(46212010),Point(46220010),Point(46225010),Point(46321010),Point(47201140),Point(47206020),Point(47209080),Point(47211050),Point(47301950)
0,6.0,7.0,7.0,4.0,7.0,5.0,4.0,6.0,5.0,5.0,...,15.0,9.0,9.0,13.0,9.0,0.0,0.0,0.0,0.0,15.0
1,8.0,5.0,9.0,1.0,3.0,6.0,6.0,5.0,5.0,1.0,...,14.0,10.0,9.0,13.0,8.0,0.0,0.0,0.0,0.0,17.0
2,4.0,5.0,9.0,3.0,6.0,4.0,5.0,6.0,6.0,3.0,...,13.0,10.0,9.0,12.0,8.0,0.0,0.0,0.0,0.0,18.0
3,4.0,3.0,10.0,1.0,6.0,3.0,6.0,7.0,4.0,3.0,...,10.0,10.0,9.0,10.0,10.0,0.0,0.0,0.0,0.0,16.0
4,6.0,5.0,10.0,-2.0,5.0,4.0,6.0,6.0,4.0,5.0,...,9.0,8.0,8.0,10.0,9.0,0.0,0.0,0.0,0.0,17.0


### iv) Install the PAMI repository

Already installed above with `!pip install -U PAMI`.

Per **Chapter 3** of the PAMI manual ([link](https://udaylab.github.io/PAMI/manuals/index.html)), PAMI expects input data either as a **dense DataFrame** (rows = transactions/timestamps, columns = items/sensors, cell = numeric reading) or already as a **transactional database** (one transaction/row of item names per line). Our cleaned `df_clean` is exactly the dense-DataFrame format PAMI expects for the conversion utility used next.


### v) Dense DataFrame → Transactional Database

We use PAMI's `denseDF2DB` converter. A sensor is included in a given timestamp's transaction if its PM2.5 reading satisfies the condition **`>= 15`** (i.e. the air was heavily polluted at that sensor, at that time).


In [ ]:
from PAMI.extras.convert import denseDF2DB as db

obj = db.denseDF2DB(df_clean)

# condition ">=", threshold = 15
obj.convert2TransactionalDatabase('PM24HeavyPollutionRecordingSensors.csv', '>=', 15)

print("Transactional database saved as:", obj.getFileName())


Transactional database saved as: PM24HeavyPollutionRecordingSensors.csv


### vi) Inspect the transactional database

Each line of `PM24HeavyPollutionRecordingSensors.csv` is one timestamp (transaction), listing every sensor (`Point(sensorID)`) whose PM2.5 reading was **≥ 15** at that time. This transactional database therefore represents *the sensors where people were frequently exposed to harmful levels of air pollution* — every line is a "snapshot" of which locations were unsafe at once.


In [ ]:
with open('PM24HeavyPollutionRecordingSensors.csv') as f:
    lines = f.readlines()

print(f"Number of transactions (timestamps): {len(lines)}")
print("\nFirst transaction (truncated):")
print(lines[0][:300], "...")


Number of transactions (timestamps): 716

First transaction (truncated):
Point(01205020)	Point(04202060)	Point(04206010)	Point(04211010)	Point(04406010)	Point(04421010)	Point(04562020)	Point(05206060)	Point(08217010)	Point(09202010)	Point(10207010)	Point(10208010)	Point(11110010)	Point(11201030)	Point(11201040)	Point(11201050)	Point(11201510)	Point(11203040)	Point(112030 ...


## 1.c) Knowledge discovery

### i) & ii) Chapters 5 & 6 of the PAMI manual

Chapter 5 covers **frequent pattern mining** concepts (support, the downward-closure/Apriori property, and algorithms such as Apriori, ECLAT and FP-Growth). Chapter 6 covers how PAMI structures its **algorithm implementations** (basic usage pattern: instantiate the algorithm class with the input file and `minSup`, call `.mine()`, then `.getPatterns()` / `.save()`).

### iii) FP-Growth implementation

FP-Growth compresses the transactional database into an **FP-tree** and mines frequent itemsets from it without candidate generation, which makes it much faster than Apriori on dense data like ours (each transaction here contains ~150–250 out of 1083 possible sensors).

**Choosing `minSup`:** we first checked how often the single *most frequent* sensor was polluted — its support tops out at about **33%** of timestamps, so a very high `minSup` (e.g. 0.5, 0.95) yields **zero** patterns. After testing a few values, **`minSup = 0.10` (10%)** was chosen: it is low enough to surface interesting multi-sensor co-occurrence patterns (patterns up to length 9), while still keeping the pattern count and runtime very manageable (~13,000 patterns, mined in about 1 second).


In [ ]:
from PAMI.frequentPattern.basic import FPGrowth as alg

minSup = 0.10  # 10% of the 716 transactions, chosen after testing 0.95 / 0.2 / 0.15 / 0.1

obj = alg.FPGrowth(iFile='PM24HeavyPollutionRecordingSensors.csv', minSup=minSup, sep='\t')
obj.mine()

frequentPatterns = obj.getPatterns()
print("Total number of Frequent Patterns:", len(frequentPatterns))

print("Total Memory in USS:", obj.getMemoryUSS())
print("Total Memory in RSS:", obj.getMemoryRSS())
print("Total ExecutionTime (s):", obj.getRuntime())


Frequent patterns were generated successfully using frequentPatternGrowth algorithm
Total number of Frequent Patterns: 13657
Total Memory in USS: 207106048
Total Memory in RSS: 230117376
Total ExecutionTime (s): 1.459324598312378


### iv) Save the patterns to `frequentPatterns.txt`

In [ ]:
obj.save('frequentPatterns.txt')
print("Patterns saved to frequentPatterns.txt")

# Quick peek
with open('frequentPatterns.txt') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())


Patterns saved to frequentPatterns.txt
Point(13113010):72
Point(21201010):72
Point(40107040):72
Point(42202520):72
Point(32203040):72


## 1.d) Visualization: longest pattern on an OpenStreetMap

We now:

1. Read `frequentPatterns.txt`.
2. Identify the **longest** pattern (the one involving the most sensors).
3. Look up each sensor's latitude/longitude in `stationInfo.csv`.
4. Plot the sensors on an OpenStreetMap using `plotly.express`.


In [ ]:
import pandas as pd
import re

# --- Load station coordinates -------------------------------------------
station_info = pd.read_csv('stationInfo.csv')
station_info['stationId_str'] = station_info['stationId'].astype(str).str.zfill(8)

def parse_point(wkt):
    # Parse a 'Point(lon lat)' WKT-style string into (lon, lat) floats.
    m = re.match(r'Point\(([-\d.]+)\s+([-\d.]+)\)', str(wkt))
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

station_info[['lon', 'lat']] = station_info['Location'].apply(
    lambda s: pd.Series(parse_point(s))
)
coord_lookup = station_info.set_index('stationId_str')[['lat', 'lon']].to_dict('index')

print(f"Loaded coordinates for {len(coord_lookup)} stations")


Loaded coordinates for 1747 stations


In [ ]:
# --- Find the longest pattern in frequentPatterns.txt --------------------
longest_items, longest_support = [], 0

with open('frequentPatterns.txt') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        left, support = line.rsplit(':', 1)
        items = left.strip().split('\t')
        if len(items) > len(longest_items):
            longest_items = items
            longest_support = int(support)

print(f"Longest pattern has {len(longest_items)} sensors, support = {longest_support}")
print(longest_items)


Longest pattern has 9 sensors, support = 72
['Point(34105010)', 'Point(34201010)', 'Point(34201020)', 'Point(34105020)', 'Point(34201250)', 'Point(34201510)', 'Point(34105510)', 'Point(34201520)', 'Point(34104510)']


In [ ]:
# --- Build a DataFrame of the longest pattern's sensor locations ---------
sensor_ids = [re.match(r'Point\((\d+)\)', it).group(1) for it in longest_items]

rows = []
for sid in sensor_ids:
    if sid in coord_lookup:
        rows.append({
            'sensor': f'Point({sid})',
            'stationId': sid,
            'lat': coord_lookup[sid]['lat'],
            'lon': coord_lookup[sid]['lon'],
        })
    else:
        print(f"[WARN] No coordinates found for station {sid} — skipped")

map_df = pd.DataFrame(rows)
map_df


,sensor,stationId,lat,lon
0,Point(34105010),34105010,34.451845,132.471653
1,Point(34201010),34201010,34.412396,132.450761
2,Point(34201020),34201020,34.378630,132.468051
3,Point(34105020),34105020,34.467397,132.401213
4,Point(34201250),34201250,34.372104,132.384418
5,Point(34201510),34201510,34.395285,132.458493
6,Point(34105510),34105510,34.454643,132.471780
7,Point(34201520),34201520,34.380967,132.469696
8,Point(34104510),34104510,34.391978,132.423487


In [ ]:
import plotly.express as px

fig = px.scatter_map(
    map_df,
    lat='lat',
    lon='lon',
    hover_name='sensor',
    zoom=8,
    height=650,
    title=(f"Longest Frequent Pattern ({len(longest_items)} sensors, "
           f"support={longest_support}) — Sensors with co-occurring "
           f"heavy PM2.5 pollution (>=15)")
)
fig.update_traces(marker=dict(size=14, color='crimson'))
fig.update_layout(map_style="open-street-map", margin=dict(l=0, r=0, t=40, b=0))
fig.show()
